# HPO LunarLander

Optuna hyperparameter optimization for `VectorTrainer` on LunarLander.

The happy path on Colab is labeled HP1 to HP4.

Hardware: L4 GPU is the intended runtime.

## Setup

### Runtime setup

Set up the runtime by running exactly one code cell in this section.

#### Colab

In [ ]:
# HP1
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys

from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.lunar_lander.logging import configure_file_logging

study_dir = Path("/content/drive/MyDrive/rl_lab/hpo")
drive.mount("/content/drive")
study_dir.mkdir(parents=True, exist_ok=True)
configure_file_logging(study_dir)

HPO_STUDY_DIR = study_dir

#### Local

In [ ]:
# Local setup
from pathlib import Path
import sys
sys.path.insert(0, "../../dqn/src")
sys.path.insert(0, "../src")
HPO_STUDY_DIR = Path("../runs")
HPO_STUDY_DIR.mkdir(parents=True, exist_ok=True)


## Imports

In [ ]:
# HP2
import torch

from dqn.vector_training import VectorTrainingConfig
from hpo.evaluation.reporting import show_lander_live_progress
from hpo.lunar_lander.study import neighbors, run_study, select_robust_best

NUM_EPISODES = 600
SCORE_WINDOW = 100
NUM_ENVS = 16
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Optimization

### Define SearchSpaces

In [ ]:
# HP3
BATCH_SIZES = [256, 512, 1024]
LEARNING_STARTS = [2_500, 5_000, 10_000]
OPTIMIZE_EVERY = [2, 4, 8]
REPLAY_CAPACITIES = [50_000, 100_000, 200_000]


BASELINE_PARAMS = {
    "learning_rate": 3e-4,
    "batch_size": 512,
    "eps_end": 0.05,
    "eps_decay": 25_000,
    "learning_starts": 5_000,
    "optimize_every": 4,
    "replay_memory_capacity": 100_000,
}


def vector_config(
    *,
    num_episodes,
    learning_rate,
    batch_size,
    eps_end,
    eps_decay,
    learning_starts,
    optimize_every,
):
    return VectorTrainingConfig(
        num_episodes=num_episodes,
        batch_size=batch_size,
        gamma=0.99,
        eps_start=1.0,
        eps_end=eps_end,
        eps_decay=eps_decay,
        tau=0.005,
        learning_rate=learning_rate,
        learning_starts=learning_starts,
        optimize_every=optimize_every,
    )


class SearchSpace0:
    def replay_memory_capacity(self, trial):
        return BASELINE_PARAMS["replay_memory_capacity"]

    def training_config(self, trial, num_episodes):
        return vector_config(
            num_episodes=num_episodes,
            learning_rate=BASELINE_PARAMS["learning_rate"],
            batch_size=BASELINE_PARAMS["batch_size"],
            eps_end=BASELINE_PARAMS["eps_end"],
            eps_decay=BASELINE_PARAMS["eps_decay"],
            learning_starts=BASELINE_PARAMS["learning_starts"],
            optimize_every=BASELINE_PARAMS["optimize_every"],
        )


class SearchSpace1:
    def replay_memory_capacity(self, trial):
        return 100_000

    def training_config(self, trial, num_episodes):
        return vector_config(
            num_episodes=num_episodes,
            learning_rate=trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True),
            batch_size=trial.suggest_categorical("batch_size", BATCH_SIZES),
            eps_end=0.05,
            eps_decay=25_000,
            learning_starts=trial.suggest_categorical("learning_starts", LEARNING_STARTS),
            optimize_every=trial.suggest_categorical("optimize_every", OPTIMIZE_EVERY),
        )


class SearchSpace2:
    def __init__(self, best_s1):
        self.best_s1 = best_s1

    def replay_memory_capacity(self, trial):
        return 100_000

    def training_config(self, trial, num_episodes):
        return vector_config(
            num_episodes=num_episodes,
            learning_rate=self.best_s1["learning_rate"],
            batch_size=self.best_s1["batch_size"],
            eps_end=trial.suggest_float("eps_end", 0.01, 0.10),
            eps_decay=trial.suggest_int("eps_decay", 10_000, 60_000, log=True),
            learning_starts=self.best_s1["learning_starts"],
            optimize_every=self.best_s1["optimize_every"],
        )


class SearchSpace3:
    def __init__(self, best_s1, best_s2):
        self.best_s1 = best_s1
        self.best_s2 = best_s2

    def replay_memory_capacity(self, trial):
        return trial.suggest_categorical("replay_memory_capacity", REPLAY_CAPACITIES)

    def training_config(self, trial, num_episodes):
        return vector_config(
            num_episodes=num_episodes,
            learning_rate=self.best_s1["learning_rate"],
            batch_size=self.best_s1["batch_size"],
            eps_end=self.best_s2["eps_end"],
            eps_decay=self.best_s2["eps_decay"],
            learning_starts=self.best_s1["learning_starts"],
            optimize_every=self.best_s1["optimize_every"],
        )


class SearchSpace4:
    def __init__(self, best_s1, best_s2, best_s3):
        self.best_s1 = best_s1
        self.best_s2 = best_s2
        self.best_s3 = best_s3

    def replay_memory_capacity(self, trial):
        return self.best_s3["replay_memory_capacity"]

    def training_config(self, trial, num_episodes):
        eps_end = self.best_s2["eps_end"]
        eps_decay = self.best_s2["eps_decay"]
        return vector_config(
            num_episodes=num_episodes,
            learning_rate=trial.suggest_float(
                "learning_rate",
                self.best_s1["learning_rate"] / 2,
                self.best_s1["learning_rate"] * 2,
                log=True,
            ),
            batch_size=trial.suggest_categorical(
                "batch_size",
                neighbors(self.best_s1["batch_size"], BATCH_SIZES),
            ),
            eps_end=trial.suggest_float(
                "eps_end",
                max(0.01, eps_end - 0.02),
                min(0.10, eps_end + 0.02),
            ),
            eps_decay=trial.suggest_int(
                "eps_decay",
                max(1, eps_decay // 2),
                eps_decay * 2,
                log=True,
            ),
            learning_starts=trial.suggest_categorical(
                "learning_starts",
                neighbors(self.best_s1["learning_starts"], LEARNING_STARTS),
            ),
            optimize_every=trial.suggest_categorical(
                "optimize_every",
                neighbors(self.best_s1["optimize_every"], OPTIMIZE_EVERY),
            ),
        )


### Optimize

In [ ]:
# HP4
def run_lander_study(study_name, search_space, n_trials, search_space_factory, previous_studies=()):
    def show_progress(study, *, target_trials):
        show_lander_live_progress(
            study,
            target_trials=target_trials,
            lander_studies=[*previous_studies, study],
        )

    study = run_study(
        study_name=study_name,
        search_space=search_space,
        n_trials=n_trials,
        num_episodes=NUM_EPISODES,
        score_window=SCORE_WINDOW,
        study_dir=HPO_STUDY_DIR,
        device=device,
        num_envs=NUM_ENVS,
        seed=SEED,
        progress_fn=show_progress,
    )
    best_params = robust_best(study, search_space_factory)
    show_progress(study, target_trials=n_trials)
    return study, best_params


def robust_best(study, search_space_factory):
    return select_robust_best(
        study=study,
        search_space_factory=search_space_factory,
        num_episodes=NUM_EPISODES,
        score_window=SCORE_WINDOW,
        device=device,
        num_envs=NUM_ENVS,
        base_seed=SEED,
    )


study0, _ = run_lander_study("s0_baseline", SearchSpace0(), 1, SearchSpace0)

study1, best_s1 = run_lander_study(
    "s1_update_economy",
    SearchSpace1(),
    40,
    SearchSpace1,
    previous_studies=[study0],
)

study2, best_s2 = run_lander_study(
    "s2_exploration",
    SearchSpace2(best_s1),
    40,
    lambda: SearchSpace2(best_s1),
    previous_studies=[study0, study1],
)

study3, best_s3 = run_lander_study(
    "s3_replay_capacity",
    SearchSpace3(best_s1, best_s2),
    10,
    lambda: SearchSpace3(best_s1, best_s2),
    previous_studies=[study0, study1, study2],
)

study4, best_s4 = run_lander_study(
    "s4_joint_finetune",
    SearchSpace4(best_s1, best_s2, best_s3),
    30,
    lambda: SearchSpace4(best_s1, best_s2, best_s3),
    previous_studies=[study0, study1, study2, study3],
)


## Analysis

Score dependence on the hyperparameters optimized in each study.

In [ ]:
# Run only after the runtime environment has been reset.
import optuna

def load_study(name):
    return optuna.load_study(
        study_name=name,
        storage=f"sqlite:///{HPO_STUDY_DIR / f'{name}.db'}",
    )

study1 = load_study("s1_update_economy")
study2 = load_study("s2_exploration")
study3 = load_study("s3_replay_capacity")
study4 = load_study("s4_joint_finetune")


In [ ]:
from IPython.display import display
from optuna.visualization import plot_contour, plot_slice

print("S1: Update economy")
display(plot_contour(study1, target_name="Score"))
print("S2: Exploration")
display(plot_contour(study2, target_name="Score"))
print("S3: Replay capacity")
display(plot_slice(study3, target_name="Score"))
print("S4: Joint fine-tune")
display(plot_contour(study4, target_name="Score"))
